**Table of contents**<a id='toc0_'></a>    
- [Fluopy trials notebook](#toc1_)    
  - [Importing all modules](#toc1_1_)    
  - [Routine template](#toc1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Fluopy trials notebook](#toc0_)
This notebook is dedicated to test functionality of fluopy

## <a id='toc1_1_'></a>[Importing all modules](#toc0_)

In [12]:
import warnings

import fluopy.fluorophores as fl
import fluopy.simulation as si
import fluopy.transitions as tr

%load_ext autoreload
%autoreload 2


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"


warnings.formatwarning = custom_warning_format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## <a id='toc1_2_'></a>[Routine template](#toc0_)

In [77]:
fluorophores = fl.construct_fluorophores(
    name="cy5_dna", distance=3, count=4, shape="square"
)
fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)

transitions = fluorophore_system.load_transitions(
    summarize=True,
    irradiance=5,
    wavelength=640,
    bleaching=False,
    energy_transfer=True,
    dstorm=True,
    dstorm_parameters={"reducing_agent": "mea", "concentration": 100, "ph": 7.5},
    energy_transfer_parameters={"overwrite": {"off": [1, 0.00001]}},
)
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set = transition_set.adjust_rates({8: 5e1})
transition_set.finalize()

transition_set.transition_df

transition_type  \
Fluorophore                         identity                                           
cy5_dna                             0                      TransitionType.EXCITATION   
                                    1            TransitionType.FLUORESCENT_EMISSION   
                                    2         TransitionType.INTERSYSTEM_CROSSING_ST   
                                    3                   TransitionType.ISOMERIZATION   
                                    4                        TransitionType.ADDUCT_T   
                                    5                        TransitionType.ADDUCT_S   
                                    6                      TransitionType.RAD_ESCAPE   
                                    7                       TransitionType.RAD_RELAX   
                                    8               TransitionType.S1_S0_TRANSITIONS   
                                    9               TransitionType.T1_S0_TRANSITIONS   
                                    10             TransitionType.CIS_S0_TRANSITIONS   
                                    11             TransitionType.OFF_S0_TRANSITIONS   
D: cy5_dna, A: cy5_dna, dist: 3.0   12                     TransitionType.CIS_FRET_1   
                                    13                     TransitionType.CIS_FRET_2   
                                    14                     TransitionType.OFF_FRET_1   
                                    15                     TransitionType.OFF_FRET_2   
                                    16                           TransitionType.FRET   
                                    17               TransitionType.S_S_ANNIHILATION   
                                    18               TransitionType.S_T_ANNIHILATION   
D: cy5_dna, A: cy5_dna, dist: 4.243 19                     TransitionType.CIS_FRET_1   
                                    20                     TransitionType.CIS_FRET_2   
                                    21                     TransitionType.OFF_FRET_1   
                                    22                     TransitionType.OFF_FRET_2   
                                    23                           TransitionType.FRET   
                                    24               TransitionType.S_S_ANNIHILATION   
                                    25               TransitionType.S_T_ANNIHILATION   

                                             abbreviation       initial_state  \
Fluorophore                         identity                                    
cy5_dna                             0                 EXC      SingleState.S0   
                                    1                 FLU      SingleState.S1   
                                    2              ISC_ST      SingleState.S1   
                                    3                 ISO      SingleState.S1   
                                    4              PET_TO      SingleState.T1   
                                    5              PET_SO      SingleState.S1   
                                    6              PET_TR      SingleState.T1   
                                    7                 OXI       SingleState.R   
                                    8             S1S0SUM      SingleState.S1   
                                    9             T1S0SUM      SingleState.T1   
                                    10           cisS0SUM     SingleState.cis   
                                    11           OFFS0SUM     SingleState.OFF   
D: cy5_dna, A: cy5_dna, dist: 3.0   12              CET_1  PairedState.S1_Cis   
                                    13              CET_2  PairedState.S1_Cis   
                                    14              OET_1  PairedState.S1_OFF   
                                    15              OET_2  PairedState.S1_OFF   
                                    16               FRET   PairedState.S1_S0   
                                    17                SSA   Paire

In [ ]:
sim = si.Simulation(transition_set)
sim.run(size=4e6, kap_sq_var=True, seed=42, use_memmap=None)

WARNING for line:         eval_floating_point_precision_error(
 Floating point precision error warning:
 The higher limit of smallest increment with a probability of 1.00e-03 is 9.59e-16.
 This was estimated using the highest possible rate which occurs for example in state combination [1, 1, 1, 1].
 Everything drawn below this number will be rounded to zero starting somewhere between 1.00e+00 - 1.00e+01. 
